# 💎 Building an Obsidian Agent 
#### 5-day AI Agents Intensive Course with Google Capstone

**Welcome to my Project!**

In this notebook, we will apply the concepts learned during the AI Agents Course to build a practical, functional agent.

**The Goal?** Build an **Obsidian Agent** capable of managing your knowledge base and assisting with Obsidian-related queries.

**What is Obsidian?** Obsidian is a popular knowledge base tool that works on local Markdown (`.md`) files. 

Our agent will act as an expert assistant that can:

1.  **Manage Notes:** Create and read markdown files in a simulated "vault".
2.  **Provide Guidance:** Search documentation to help you use Obsidian features.
3.  **Be Observable:** We will implement logging to observe exactly what the agent thinks and does.
4.  **Be Reliable:** We will evaluate if the agent actually performed the task correctly.

## 🗺️ Roadmap
- **Setup**: Configure environment and dummy vault.
- **Tools**: Build tools for file manipulation and documentation search.
- **Agents**: Create the agents and multi-agent interactions.
- **Session and Memory**: Implement runtime configuration.
- **Observability**: Logging, tracing, and monitoring.
- **Execution**: Run complex tasks.
- **Evaluation**: Verify the results.

## ⚙️ Section 1: Setup and Configuration

First, let's set up our environment. We need the **Agent Development Kit (ADK)**, a Gemini API key, and an Obsidian Vault to test the agent. We will create a temporary directory to act as our "Obsidian Vault".

### 1.1: Install dependencies

The Kaggle Notebooks environment includes a pre-installed version of the [google-adk](https://google.github.io/adk-docs/) library for Python and its required dependencies, so you don't need to install additional packages in this notebook.

To install and use ADK in your own Python development environment, you can do so by running:

```bash
pip install google-adk # Recommended in an isolated environment

uv add google-adk # If using `uv` for project management

```

### 1.2: Configure your Gemini API Key 🔑

This notebook uses the [Gemini API](https://ai.google.dev/gemini-api/), which requires an API key.

**1. Get your API key**

If you don't have one already, create an [API key in Google AI Studio](https://aistudio.google.com/app/api-keys).

**2. Add the key to Kaggle Secrets**

Next, you will need to add your API key to your Kaggle Notebook as a Kaggle User Secret.

1. In the top menu bar of the notebook editor, select `Add-ons` then `Secrets`.
2. Create a new secret with the label `GOOGLE_API_KEY`.
3. Paste your API key into the "Value" field and click "Save".
4. Ensure that the checkbox next to `GOOGLE_API_KEY` is selected so that the secret is attached to the notebook.

**3. Authenticate in the notebook**

Run the cell below to access the `GOOGLE_API_KEY` you just saved and set it as an environment variable for the notebook to use:

Alternatively, if you are running this notebook in your local environment, ensure that you have set an environment variable with your API key named `GOOGLE_API_KEY` (or `GEMINI_API_KEY`).

In [1]:
# Import os module to handle environment variables
import os

# Standardize API Key for ADK (expects GOOGLE_API_KEY)
if "GOOGLE_API_KEY" not in os.environ:
    # 1. Try Local GEMINI_API_KEY
    if "GEMINI_API_KEY" in os.environ:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
        print("✅ Google API Key loaded from GEMINI_API_KEY.")
    
    # 2. Try Kaggle Secrets
    else:
        try:
            from kaggle_secrets import UserSecretsClient # type: ignore
            os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
            print("✅ API Key loaded from Kaggle Secrets.")
        except ImportError:
            print("⚠️ Warning: 'kaggle_secrets' not found. Ensure GOOGLE_API_KEY or GEMINI_API_KEY is set.")
        except Exception as e:
            print(f"⚠️ Warning: Failed to load secret. {e}")
else:
    print("✅ Google API Key found in environment.")

# Force GenAI to use API Key mode (not Vertex AI)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

✅ API Key loaded from Kaggle Secrets.


### 1.3 Import ADK components

Now, import the specific components you'll need from the Agent Development Kit and the Generative AI library. Then, we initialize the models we'll be using. This keeps your code organized and ensures we have access to the necessary building blocks.

In [2]:
from google.adk.agents import LlmAgent, LoopAgent, SequentialAgent
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.google_llm import Gemini
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner, InMemoryRunner
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import google_search, load_memory, preload_memory, AgentTool, ToolContext
from google.genai import types # Ensure types available (e.g. `types.Markdown`)
print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4 Create an Obsidian Vault and Config User Settings

We will create a temporary directory to simulate an Obsidian vault where any md files created by the agent will be stored. The path could be changed to point to an actual Obsidian vault on your local machine if you want to test it with your own notes.

In [3]:
# Setup Dummy Vault
VAULT_DIR = "obsidian_vault_dummy"

# Create Vault if not exists
if not os.path.exists(VAULT_DIR):
    os.makedirs(VAULT_DIR)

print(f"✅ Environment ready. Vault ready at: {VAULT_DIR}/")

✅ Environment ready. Vault ready at: obsidian_vault_dummy/


In [4]:
# Define a constant for the user name
USER_NAME = "Pau"
print(f"✅ User name set to: {USER_NAME}")

✅ User name set to: Pau


### 1.5: Configure Models and Retry Options

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [5]:
# Define retry configuration for Gemini models
retry_config = types.HttpRetryOptions(
    attempts=3,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [6]:
# Configure standard models for use within ADK
model_flash_lite = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)
model_flash = Gemini(model="gemini-2.5-flash", retry_options=retry_config)
model_pro = Gemini(model="gemini-2.5-pro", retry_options=retry_config)

## 🛠️ Section 2: Tools

An agent is only as good as its tools. Since Obsidian works with local Markdown files, our agents must be able to read and write files to the disk.

For the Obsidian Agent we'll utilize two types of tools:

**1. Function Tools:**

    - **File Manipulation Tools:** To create and read notes in the vault.
    - **Folder Management Tools:** To organize notes within the vault.

Note: A folder management system would let us manage a vault like a librarian and build collections. To keep this prototype simple we'll defer this feature for next steps, and our application will act as a simple bucket system, placing all documents in the same directory.

**2. Tool Agents:**

    - **Content Tools:** To manage content within the notes.
    - **Formatting Tools:** Additional tools to apply formatting to the notes.

Note: Obsidian includes many more features like links, graph view, and those provided by plugins. For this capstone, we will focus on basic text operations to keep things manageable.

### 2.1: Function Tools

#### File Operations

We will define four core functions to interact with the Obsidian vault:

1. `create_note`: To make new notes.
2. `read_note`: To read existing content.
3. `update_note`: To modify existing content.
4. `delete_note`: To remove notes.
5. `list_notes`: To see all notes in the vault.
6. `append_to_note`: To add content to an existing note without overwriting.
7. `merge_notes`: To combine content from two notes into one.

#### Tools availability

```mermaid
flowchart LR
    direction LR
    User[User] --> Root[Root] --> Tools[Function Tools]
    subgraph Agents[Agents]
        Root --- SubAgents[Sub Agents]
        Root --- ToolAgents[Tool Agents]
    end
    direction LR
    ToolAgents[Tool Agents] --> Tools[Function Tools]
    SubAgents[Sub Agents] --> Tools[Function Tools]

    subgraph Tools[Available Tools]

    direction TB
        CreateNote[create_note]
        ReadNote[read_note]
        UpdateNote[update_note]
        DeleteNote[delete_note]
        ListNotes[list_notes]
        AppendToNote[append_to_note]
        MergeNotes[merge_notes]
    end
    
    
    style Root fill:#4a90e2,color:#fff
    style ToolAgents fill:#50c878,color:#fff
    style SubAgents fill:#9b59b6,color:#fff
```

In [7]:
# File Manipulation Tools

def create_note(filename: str, content: str) -> dict:
    """Creates a new markdown note in the vault.
    
    Args:
        filename: The name of the file (e.g., 'Meeting Notes.md'). 
                  If .md is missing, it will be added.
        content: The text content to write into the note.
        
    Returns:
        A dictionary indicating success or failure.
    """
    if not filename.endswith(".md"):
        filename += ".md"
        
    filepath = os.path.join(VAULT_DIR, filename)
    
    try:
        with open(filepath, "w") as f:
            f.write(content)
        return {"status": "success", "message": f"Note '{filename}' created successfully."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

def read_note(filename: str) -> dict:
    """Reads the content of an existing markdown note.
    
    Args:
        filename: The name of the file to read.
        
    Returns:
        The content of the note or an error message.
    """
    if not filename.endswith(".md"):
        filename += ".md"
        
    filepath = os.path.join(VAULT_DIR, filename)
    
    if not os.path.exists(filepath):
        return {"status": "error", "message": f"Note '{filename}' does not exist."}
        
    try:
        with open(filepath, "r") as f:
            content = f.read()
        return {"status": "success", "content": content}
    except Exception as e:
        return {"status": "error", "message": str(e)}
    
def update_note(filename: str, content: str) -> dict:
    """Updates the content of an existing markdown note.
    
    Args:
        filename: The name of the file.
        content: The new text content to write into the note.
        
    Returns:
        A dictionary indicating success or failure.
    """
    if not filename.endswith(".md"):
        filename += ".md"
        
    filepath = os.path.join(VAULT_DIR, filename)
    
    if not os.path.exists(filepath):
        return {"status": "error", "message": f"Note '{filename}' does not exist."}
        
    try:
        with open(filepath, "w") as f:
            f.write(content)
        return {"status": "success", "message": f"Note '{filename}' updated successfully."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

def delete_note(filename: str) -> dict:
    """Deletes an existing markdown note.
    
    Args:
        filename: The name of the file to delete.
        
    Returns:
        A dictionary indicating success or failure.
    """
    if not filename.endswith(".md"):
        filename += ".md"
        
    filepath = os.path.join(VAULT_DIR, filename)
    
    if not os.path.exists(filepath):
        return {"status": "error", "message": f"Note '{filename}' does not exist."}
        
    try:
        os.remove(filepath)
        return {"status": "success", "message": f"Note '{filename}' deleted successfully."}
    except Exception as e:
        return {"status": "error", "message": str(e)}
    
def list_notes() -> list[str]:
    """Lists all markdown notes in the vault.
    
    Returns:
        A list of filenames in the vault.
    """
    try:
        return [f for f in os.listdir(VAULT_DIR) if f.endswith(".md")]
    except Exception as e:
        return {"status": "error", "message": str(e)}
    

def append_to_note(filename: str, content: str) -> dict:
    """Appends content to an existing markdown note.
    
    Args:
        filename: The name of the file.
        content: The text content to append.
        
    Returns:
        A dictionary indicating success or failure.
    """
    if not filename.endswith(".md"):
        filename += ".md"
        
    filepath = os.path.join(VAULT_DIR, filename)
    
    if not os.path.exists(filepath):
        return {"status": "error", "message": f"Note '{filename}' does not exist."}
        
    try:
        with open(filepath, "a") as f:
            f.write("\n" + content)
        return {"status": "success", "message": f"Content appended to '{filename}' successfully."}
    except Exception as e:
        return {"status": "error", "message": str(e)}
    
def merge_notes(source_filename: str, target_filename: str) -> dict:
    """Merges content from source note into target note.
    
    Args:
        source_filename: The name of the source file.
        target_filename: The name of the target file.

    Returns:
        A dictionary indicating success or failure.
    """
    if not source_filename.endswith(".md"):
        source_filename += ".md"
    if not target_filename.endswith(".md"):
        target_filename += ".md"

    source_filepath = os.path.join(VAULT_DIR, source_filename)
    target_filepath = os.path.join(VAULT_DIR, target_filename)

    if not os.path.exists(source_filepath):
        return {"status": "error", "message": f"Source note '{source_filename}' does not exist."}
    if not os.path.exists(target_filepath):
        return {"status": "error", "message": f"Target note '{target_filename}' does not exist."}

    try:
        with open(source_filepath, "r") as source_file:
            source_content = source_file.read()
        with open(target_filepath, "a") as target_file:
            target_file.write("\n" + source_content)
        return {"status": "success", "message": f"Note '{source_filename}' merged into '{target_filename}' successfully."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# Aggregate File Interaction Tools
file_tools = [ 
    create_note,
    read_note,
    update_note,
    delete_note,
    list_notes,
    append_to_note,
    merge_notes,
]

print("✅ File interaction tools defined.")

✅ File interaction tools defined.


### 2.2: Tool Agents

#### Content and Formatting Tools

There are several content structuring and formatting tasks that we might need to perform when creating or editing notes, and in some cases a proper formatting is crucial for usability. As some of these capabilities may require more advanced text manipulation, we'll use LLMs to assist with these tasks. Within specific instructions and using the `AgentTool` class, **tool agents** become a toolkit for the root agent and other agents. Different models could be assigned to each agent based on the complexity of the task.

In [8]:

# Content and Formatting Tool Agents

summarize_agent = LlmAgent(
    name="SummarizeAgent",
    model=model_flash_lite,
    description="Condense text into concise summaries with `max_length` (short, medium, long, or custom lines/characters).",
    instruction="""You are a summarization specialist.
    **Guidelines**:
    - Read the provided `content` and capture the main ideas and key information.
    - Respect the `max_length` argument (short, medium, long, or custom lines/characters).
    **Task**:
    - Generate a concise summary based on the `content` and specified `max_length`.
    **Output**:
    - Return only the summary text in markdown form.
    - Do not add prefaces, metadata, or analysis beyond the summary.""",
    output_key="summary_text",
)

key_points_agent = LlmAgent(
    name="KeyPointsAgent",
    model=model_flash_lite,
    description="Extract actionable key points from source material using `num_points` distinct insights.",
    instruction="""You identify the most important points in the supplied `content`.
    **Guidelines**:
    - Extract up to `num_points` distinct insights.
    - Preserve factual accuracy and original intent.
    **Task**:
    1. Identify all key points in the content.
    2. Extract the most relevant `num_points` distinct insights from all key points in the content.
    **Output**:
    - Return a numbered markdown list of the extracted points.
    - Keep each point concise and self-contained.""",
    output_key="key_points",
)

outline_agent = LlmAgent(
    name="OutlineAgent",
    model=model_flash,
    description="Generate structured outlines from notes or research with different levels of depth.",
    instruction="""You create hierarchical outlines from the given `content`.
    **Guidelines**:
    - Respect the requested depth (e.g., 1-level, 2-level, or 3-level headings).
    - Capture the logical flow of the source material while staying faithful to the text.
    **Task**:
    - Produce an outline based on the `content` and specified depth.
    **Output**:
    - Return markdown headings (e.g., #, ##, ###, -) representing the outline only.
    - Do not include prose explanations outside of the outline.""",
    output_key="outline_text",
)

breakdown_agent = LlmAgent(
    name="BreakdownAgent",
    model=model_flash,
    description="Break complex tasks into actionable subtasks and checklists with different levels of granularity.",
    instruction="""You decompose task description inputs into smaller steps.
    **Guidelines**:
    - Interpret if granularity (e.g. high, medium, low) is requested to control detail.
    - Focus on creating manageable, ordered subtasks that cover the original request.
    **Task**:
    1. Understand the overall task description.
    2. If provided granularity, determine the level of detail needed.
    3. Break down the task into clear, actionable subtasks.
    - Generate a checklist of subtasks based on the task description and granularity.
    **Output**:
    - Return an actionable checklist in markdown using `- [ ]` syntax.
    - Keep each checkbox specific and outcome-oriented.""",
    output_key="task_breakdown",
)

lint_agent = LlmAgent(
    name="LintAgent",
    model=model_flash,
    description="Ensure markdown adheres to proper syntax.",
    instruction="""You lint markdown content.
    **Guidelines**:
    - Inspect the provided `content` for syntax errors or malformed structures.
    - Fix heading levels, code blocks, lists, and spacing as needed without altering meaning.
    **Task**:
    - Inspect and correct the markdown syntax in the provided `content`.
    **Output**:
    - Return ONLY the corrected markdown.
    - Do not add explanations or commentary.""",
    output_key="linted_markdown",
)

template_agent = LlmAgent(
    name="TemplateAgent",
    model=model_flash,
    description="Apply structured `template` to raw markdown content.",
    instruction="""You remodel `content` so it follows a given `template`.
    **Guidelines**:
    - Preserve the original ideas and facts.
    - Map sections of the content into the template structure, filling placeholders as needed.
    **Task**:
    - Apply the `template` to the `content`, ensuring all sections are appropriately filled.
    **Output**:
    - Return the fully formatted markdown following the template.
    - Exclude any reasoning steps or notes.""",
    output_key="templated_content",
)

style_agent = LlmAgent(
    name="StyleAgent",
    model=model_pro,
    description="Rewrite content so it matches a reference style file.",
    instruction="""You adapt tone, formatting, and voice to match a reference note.
    **Tools**:
    - `list_notes`: List available notes to find the style file if a file name is misspelled.
    - `read_note`: Read the style file to understand its conventions.
    **Guidelines**:
    - Follow the structure and formatting of the reference note if provided.
    - Maintain the original meaning and context while adapting the style.
    - Use `list_notes` and `read_note` to access the style file.
    **Task**:
    Case A (receive a file name):
    1. Read the example style using `read_note(style_file)`.
    2. Mirror the example's cadence, vocabulary, and markdown conventions without copying sentences 
    verbatim in the target content.
    Case B (no style file, example content provided):
    1. Read the provided `example_content` if no style file is given.
    2. Mirror the example's cadence, vocabulary, and markdown conventions without copying sentences 
    verbatim in the target content.
    Case C (style description provided):
    1. Understand the overall style description intent.
    2. Use the provided style description to adapt the content.
    **Output**:
    - Return only the rewritten content in the new style.
    - If the style file cannot be read, explain the issue and request a valid file name.""",
    tools=[list_notes, read_note],
    output_key="styled_content",
)

# Convert agents into callable tools for other agents
summarize_tool = AgentTool(agent=summarize_agent)
key_points_tool = AgentTool(agent=key_points_agent)
outline_tool = AgentTool(agent=outline_agent)
breakdown_tool = AgentTool(agent=breakdown_agent)
lint_tool = AgentTool(agent=lint_agent)
template_tool = AgentTool(agent=template_agent)
style_tool = AgentTool(agent=style_agent)

print("✅ Content and Formatting tools ready!")

✅ Content and Formatting tools ready!


#### The Review Tool

Content evaluation is a more complex task than summarization, requiring a critical perspective rather than a generative one. To handle this, we will encapsulate this logic in a ToolAgent. This agent, powered by an expert pro model, will focus on reviewing content for clarity, coherence, and completeness; improving the reliability and quality of the Obsidian Agent's output.

In [9]:
# Review tool agent
review_agent = LlmAgent(
    name="ReviewAgent",
    model=model_pro,
    description="Review, analyze, summarize, outline, or critique the content of existing notes.",
    instruction="""You are a Reviewer/Content Analyst Specialist.
    **Role**: Analyze and critique text or existing notes and suggest improvements.
    **Tools**:
    - `list_notes`: List available notes.
    - `read_note`: Read the target note.
    - `summarize_tool`: Summarize the content of the target content.
    - `key_points_tool`: Extract key points from the target content.
    - `outline_tool`: Generate an outline from the target content.
    **Guidelines**:
    - Use `list_notes` to list available notes to find the target note if needed.
    - Use `summarize_tool`, `key_points_tool` to analyze the content
    - Use `outline_tool` to create a structured review summary.
    - Suggest improvements for clarity, coherence, completeness, and organization.
    - Provide constructive feedback or structured outlines.
    **Task**: 
    1. Analyze text or existing notes.
    2. Based on your analysis, provide insights, summaries, or improvements.
    3. Return a structured critique or summary or *exactly* "No changes needed" if no changes are needed.
    **Output**:
    - Return a structured critique, summary, or outline based on the analysis.""",
    tools = [list_notes, read_note, summarize_tool, key_points_tool, outline_tool],
    output_key="review_summary"
)

review_tool = AgentTool(agent=review_agent)

#### The Fast-Track Logger

Applying the same principle as the Review Tool, we will create a tool that encapsulates the file manipulation process (create, append or update). This quick logger is designed for fast logging of changes to existing notes or creating new ones with minimal friction.

In [10]:
# Quick logger agent

quick_logger_agent = LlmAgent(
    name="QuickLogger",
    model=model_flash,
    description="Create note, append text to note or update note with provided content.",
    instruction="""You are the Quick Logger that creates, appends or updates notes.
    **Role**: 
    Your role is add the provided content to an existing note, create a new note in the user's vault, or update an existing note.
    **Tools**:
    - `list_notes`: List available notes.
    - `read_note`: Read content of an existing note.
    - `update_note`: Update content of an existing note.
    - `append_to_note`: Append content to an existing note.
    - `create_note`: Create a new note if it does not exist.
    **Guidelines**:
    - Use `list_notes` to check for existing files before creating.
    - If adding content, do not read it first; just use `append_to_note` for adding the content.
    - Use valid Markdown syntax.
    **Task**:
    Case A (new note):
    1. Identify if the target note filename exists using `list_notes`.
    2. If it does not exist, use `create_note` to make a new note with the provided content.
    Case B (append to existing note):
    1. Identify the target note filename.
    2. Extract the content to be added from the input.
    3. Format the text appropriately (e.g., add a bullet point ` - `, a checkbox ` - [ ] `, or a timestamp if it looks like a log).
    4. Use the `append_to_note` tool.
    Case C (update existing note):
    1. Identify the target note filename (using `list_notes`).
    2. Use `read_note` to extract the existing content.
    3. Extract the new content from the input.
    4. Use the `update_note` tool to update the existing content with new content.
    **Output**:
    - Return a report indicating whether a note was created, appended to, or updated.
    """,
    tools=[list_notes, read_note, append_to_note, create_note, update_note],
    output_key="logger_status"
)

# Convert to tool
log_tool = AgentTool(agent=quick_logger_agent)

print("✅ Quick Logger Agent defined.")

✅ Quick Logger Agent defined.


#### Web Tools

Our agents should also be experts on *how* to use Obsidian. Instead of hallucinating syntax, we'll give them a tool to look up documentation, the built-in Google Search Tool from ADK. The Google Search Tool can be included as a tool for a specialist agent with instructions to search only specified websites like the official Obsidian documentation site.

- **Documentation Lookup**: Allowing the agent to find and reference official Obsidian documentation.
- **Research**: Enabling the agent to gather information on various topics to include in notes.

We can create specialized tool agents that are tailored for specific tasks or domains. These agents can leverage domain knowledge and expertise to provide more accurate and relevant information to the root agent. We can choose for this tasks a reasoning model.

In [11]:
# Obsidian Documentation specialist
docs_agent = LlmAgent(
    name="DocumentationAgent",
    model=model_flash,
    description="Consult official Obsidian documentation for help with features and syntax.",
    instruction="""You are an Obsidian Documentation Specialist.
    **Tools**:
    - `google_search`: Use this tool to search the web for information.
    **Guidelines**:
    - Use `google_search` with the Obsidian Documentation site (help.obsidian.md).
    - Provide concise explanations.
    - Always cite sources.
    **Task**: Search official documentation (help.obsidian.md) to answer questions about Obsidian features, syntax, and plugins.""",
    tools=[google_search],
    output_key="obsidian_documentation"
)

# Research agent
search_agent = LlmAgent(
    name="SearchAgent", 
    model=model_pro,
    description="Search the web for general information, facts, and research topics.",
    instruction="""You are a Web Research Specialist.
    **Tools**:
    - `google_search`: Use this tool to search the web for information.
    **Guidelines**:
    - Search for accurate, up-to-date information.
    - Synthesize findings into clear, concise summaries.
    - Verify information from multiple sources when possible.
    - Cite sources.
    **Task**: Search the web for accurate information on requested topics.""",
    tools=[google_search],
    output_key="research_summary"
)

# Convert agents to tools for use by other agents
docs_tool = AgentTool(agent=docs_agent)
search_tool = AgentTool(agent=search_agent)

print("✅ Google Search Tools ready!")

✅ Google Search Tools ready!


## 🤖 Section 3: Multi-Agent Architecture

In this section, we'll implement a hierarchical multi-agent architecture, where the thinking will be handled by the `LlmAgent` agents, while workflow orchestration will be managed by `ParallelAgent`, `SequentialAgent`, and `LoopAgent` classes.

The ADK hierarchy consists of different levels:

- **Root Agent**: The main orchestrator that manages the overall workflow and delegates tasks to other agents. An LlmAgent type that interacts directly with the user. Can be any type of agent except previously defined Tool Agents.
- **Sub-Agents**: Task executors that can complete complex tasks and respond directly to users. They could be of various types depending on the task complexity. Can be both LlmAgent and Workflow agent types.

Depending on the task the agent classes utilized are:

- **Llm Agents**: Non-deterministic agents that utilize large language models to perform tasks that require understanding and generation of natural language.
- **Workflow Agents**: Deterministic agents that manage and execute specific workflows. They have sub-agents under them that execute parts of the workflow.

#### ADK Architecture Overview

```mermaid
flowchart LR
    subgraph LR Agents[Agents]
        subgraph Root[Root Agent]
        LlmAgent[LlmAgent]
        end
        subgraph  ToolAgents[Tool Agents]
        direction LR
            LlmToolAgent[LlmAgent] ---| OR | WorkflowToolAgent[WorkflowAgent]
        end
            subgraph WorkflowToolAgent[Workflow Agents]
            SubAgentToolAgent[Workflow Sub-Agents]
        end
        subgraph  SubAgents[Sub-Agents of Root Agent]
        direction LR
            LlmSubAgent[LlmAgent] ---| OR | WorkflowSubAgent[WorkflowAgent]
        end
            subgraph WorkflowSubAgent[Workflow Agents]
                SubAgentSubAgent[Workflow Sub-Agents]
            end
    end
    User[User] -->|Query| Root
    Root -->|Query| ToolAgents
    Root -->|Delegate| SubAgents
    SubAgents -->|Query| ToolAgents 

    style User fill:black,color:#fff
    style Root fill:#4a90e2,color:#fff
    style SubAgents fill:#9b59b6,color:#fff
    style ToolAgents fill:#50c878,color:#fff
    style LlmAgent fill:black,color:#fff
    style LlmToolAgent fill:black,color:#fff
    style LlmSubAgent fill:black,color:#fff 
    style WorkflowToolAgent fill:black,color:#fff
    style SubAgentToolAgent fill:#9b59b6,color:#fff
    style WorkflowSubAgent fill:black,color:#fff
    style SubAgentSubAgent fill:#9b59b6,color:#fff
    
    
```

### 3.1 The Smart Writer

We will define a workflow agent to manage the note creation and refinement process through a well-defined sequence of steps, including a refinement loop.

In [12]:
# Define loop termination phrase to keep instructions deterministic, mirroring the ADK example
COMPLETION_PHRASE = "No major issues found."

def exit_loop(tool_context: ToolContext):
  """Call only when the critic responds with the completion phrase to stop the refinement loop."""
  print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
  tool_context.actions.escalate = True
  return {}

# Initial writer produces the first draft that the loop will iteratively improve
research_draft_writer = LlmAgent(
    name="ResearchDraftWriter",
    model=model_pro,
    description="Create the first markdown draft using research context.",
    instruction="""You are the Draft Writer Specialist for the Obsidian Agent.
    **Tools**:
    - Use `SearchAgent` tool to gather research.
    - Use the `OutlineAgent` tool to plan before writing.
    - Use the `TemplateAgent` tool if the user requests a specific format (e.g., Blog, Report).
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**:
    1. Gather relevant research using the `SearchAgent` tool.
    2. Read the research summary and capture the user's intent for the note.
    3. Produce a well-structured markdown draft (headings, lists, short paragraphs) that covers the critical points.
    4. Keep the tone informative and concise so it is easy to refine later.
    **Output**:
    Return only the drafted markdown content without explanations or TODO placeholders.""",
    tools=[outline_tool, template_tool, search_tool],
    output_key="current_document",
)

# Critic agent provides actionable feedback or signals completion
refinement_critic = LlmAgent(
    name="RefinementCritic",
    model=model_pro,
    description="Critique the latest draft and signal when it is good enough.",
    instruction=f"""You review the current draft and decide if additional revisions are necessary.
    **Current Draft**
    {{current_document}}
    **Tools**
    - Use the `lint_tool` to check for grammar/syntax errors.
    - Use the `style_tool` to verify tone consistency.
    - Use the `review_tool` to gather overall feedback.
    - Use the `breakdown_tool` to identify complex sections needing simplification.
    **Task**
    - Evaluate clarity, structure, alignment with the research summary, and overall usefulness.
    - Include tool findings in your critique list.
    - If clear improvements remain, list 1-3 specific, actionable fixes in markdown bullets 
    (e.g., "Clarify the call-to-action").
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Completion**
    - If the draft already satisfies the request and no major issues remain, respond *exactly* with 
    the phrase "{COMPLETION_PHRASE}" and nothing else.
    **Output**
    - Provide either the concise critique list or the exact completion phrase. Do not include commentary beyond that.""",
    tools=[lint_tool, style_tool, review_tool, breakdown_tool],
    output_key="critique_notes",
)

# Refiner either applies the critic's feedback or exits the loop by calling the tool
refinement_writer = LlmAgent(
    name="RefinementWriter",
    model=model_pro,
    description="Apply critiques to the draft or exit when the critic signals completion.",
    instruction=f"""You revise the draft using the latest critique.
    **Current Draft**
    {{current_document}}
    **Critique / Suggestions**
    {{critique_notes}}
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**
    1. If the critique text matches "{COMPLETION_PHRASE}", call the `exit_loop` tool and output *only* 
    the document without any additional text.
    2. Otherwise, thoughtfully integrate every actionable point while preserving the document's intent and formatting.
    **Output**
    - Return only the improved markdown without summaries or justification.""",
    tools=[exit_loop],
    output_key="current_document",
)

# Publisher agent saves the final draft into the vault
publish_agent = LlmAgent(
    name="PublishAgent",
    model=model_flash_lite,
    description="Publish the draft in a new note in the user's vault.",
    instruction="""You save the current document into the user's Obsidian vault.
    **Current Document**
    {{current_document}}
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**:
    - Use the `create_note` tool to save the document.""",
    tools=[create_note],
    output_key="publication_status",
)

# LoopAgent executes the critic/writer cycle
refine_agent = LoopAgent(
    name="RefineAgent",
    description="Iteratively critique and refine the draft until the critic signals completion or max iterations are reached.",
    sub_agents=[refinement_critic, refinement_writer],
    max_iterations=3,
)

# Sequential workflow: gather research, create the initial draft, then run refinement loop, and publish
new_content_crafter = SequentialAgent(
    name="NewContentCrafter",
    description="Run research, draft the note, then refine iteratively using the critic/writer loop.",
    sub_agents=[research_draft_writer, refine_agent, publish_agent]
)

print("✅ New Content Crafter pipeline defined.")

✅ New Content Crafter pipeline defined.


### 3.2: The Wise Editor

For editing existing content, we cannot simply reuse the RefineAgent by changing its entry and exit points within a new SequentialAgent. Instead, we need to replicate the functionality of the `RefineAgent`, and also create an initial drafter that starts by reading an existing note instead of doing research and drafting from scratch. This workflow involves loading the existing content, making the necessary edits, and then saving the updated content back to the note. This task will involve a new publishing step.

In [13]:
# The document loader just loads existing content and instructions for edits into state variables
load_docs_agent = LlmAgent(
    name="LoadDocsAgent",
    model=model_flash_lite,
    description="Load an existing note and interpret edit instructions.",
    instruction="""You load text from files and output it.
    **Tools**:
    - `read_note`: Read the content of an existing note.
    - `list_notes`: List available notes to find the target note if needed.
    **Guidelines**:
    - Use `list_notes` to verify the filename if there are typos.
    - You don't need to summarize or analyze the content here; just load it using `read_note`.
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**:
    1. Identify the file the user wants to edit.
    2. Call `read_note` to get the content.
    3. Output the content into `existing_document` with the following format:
    ```Instructions:
    {user's edit instructions here including the filename}
    Existing Document:
    {file content here}
    ```
    **Output**:
    - Return *only* the `existing_document` and `instructions`.""",
    tools=[read_note, list_notes], # Needs list_notes in case of typo in filename
    output_key="existing_document"
)

# The document rewriter receives the existing content and instructions to produce the first draft
document_rewriter = LlmAgent(
    name="DocumentRewriter",
    model=model_pro,
    description="Create the first markdown draft using research context.",
    instruction="""You are the Document Rewriter Specialist for the Obsidian Agent.
    **Context**
    - Existing file: {{existing_document}}
    **Tools**
    - SearchAgent: to gather research.
    - DocsAgent: to consult Obsidian documentation if needed.
    - StyleAgent: to match user tone preferences of existing document.
    - OutlineAgent: to plan before writing.
    - TemplateAgent: if the user requests a specific format (e.g., Blog, Report).
    **Guidelines**:
    - Preserve the original intent of the document while improving clarity and structure.
    - Keep the tone consistent with the existing content (using the 'StyleAgent').
    - Keep the tone informative and concise so it is easy to refine later.
    - Maintain the technical detailing and accuracy of the original document.
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**
    1. Read the existing document and understand its content.
    2. Read your instructions and capture the user's intent for the note.
    3. If your research is related to Obsidian, use `docs_tool` to gather specific Obsidian context.
    4. Gather more information using the `search_tool` if needed.
    5. Use the `outline_tool` to structure the gathered information.
    6. Produce a well-structured markdown draft (headings, lists, short paragraphs).
    **Output**
    - Return only the drafted markdown content without explanations or TODO placeholders.""",
    tools=[search_tool, docs_tool, style_tool, outline_tool, template_tool],
    output_key="current_document",
)

# Critic agent provides actionable feedback or signals completion
refinement_edit_critic = LlmAgent(
    name="RefinementEditCritic",
    model=model_pro,
    description="Critique the latest draft and signal when it is good enough.",
    instruction=f"""You review the current draft and decide if additional revisions are necessary.
    **Current Draft**
    {{current_document}}
    **Tools**
    - `LintAgent`: to check for grammar/syntax errors.
    - `StyleAgent`: to verify tone consistency.
    - `ReviewAgent`: to gather overall feedback.
    - `BreakdownAgent`: to understand complex sections needing simplification.
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**
    - Evaluate clarity, structure, alignment with the research summary, and overall usefulness.
    - Include tool findings in your critique list.
    - If clear improvements remain, list 1-3 specific, actionable fixes in markdown bullets 
      (e.g., "Clarify the call-to-action").
    **Completion**
    - If the draft already satisfies the request and no major issues remain, respond *exactly* with
      the phrase "{COMPLETION_PHRASE}" and nothing else.
    **Output**
    - Provide either the concise critique list or the exact completion phrase. Do not include commentary beyond that.""",
    tools=[lint_tool, style_tool, review_tool, breakdown_tool],
    output_key="critique_notes",
)

# Refiner either applies the critic's feedback or exits the loop by calling the tool
refinement_edit_writer = LlmAgent(
    name="RefinementEditWriter",
    model=model_pro,
    description="Apply critiques to the draft or exit when the critic signals completion.",
    instruction=f"""You revise the draft using the latest critique.
    **Current Draft**
    {{current_document}}
    **Critique / Suggestions**
    {{critique_notes}}
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**
    1. If the critique text matches "{COMPLETION_PHRASE}", call the `exit_loop` tool and output *only* the document without any additional text.
    2. Otherwise, thoughtfully integrate every actionable point while preserving the document's intent and formatting.
    **Output**
    - Return only the improved markdown without summaries or justification.""", 
    tools=[exit_loop],
    output_key="current_document",
)

# LoopAgent executes the critic/writer cycle
refine_edit_agent = LoopAgent(
    name="RefineEditAgent",
    description="Iteratively critique and refine the draft until the critic signals completion or max iterations are reached.",
    sub_agents=[refinement_edit_critic, refinement_edit_writer],
    max_iterations=3,
)


# The updater agent saves the final draft back to the file
updater_agent = LlmAgent(
    name="UpdaterAgent",
    model=model_flash,
    description="Save changes to the existing note.",
    instruction="""Use `update_note` to save the `current_document` edited document back to the file.
    **Current Document**
    {{current_document}}
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    **Task**:
    - Identify the correct file to update with `list_notes`.
    - Use the `update_note` tool to save the document.""",
    tools=[update_note, list_notes],
    output_key="update_status"
)

# The sequence of loading, rewriting, refining, and updating
existing_content_editor = SequentialAgent(
    name="ExistingContentEditor",
    description="Load existing file content, rewrite and refine iteratively, and save changes.",
    sub_agents=[load_docs_agent, document_rewriter, refine_edit_agent, updater_agent]
)

print("✅ Existing Content Editor ready!")

✅ Existing Content Editor ready!


### 3.3 Specialized Sub-Agents

The sub-agents of root agent are agents designed to handle specific tasks or workflows within the broader context of the Obsidian Agent. Each sub-agent will have its own set of tools and instructions tailored to its function.

We'll create three specialized sub-agents:

- **Task Planning Agent**: Breaks down complex tasks into manageable steps.
- **Context Files Specialist**: Manages and creates dedicated context files for LLMs.
- **VSCode Specialist**: Assists with creating and managing VSCode custom agent mode files.

In [14]:

# Task planning agent

task_plan_agent = LlmAgent(
    name="TaskPlanner",
    model=model_pro,
    description="Decompose complex user requests into structured, actionable plans documented in the vault.",
    instruction="""You are a Task Planner specialist. 
    **Role**: Your goal is to decompose complex user requests into actionable, structured plans and document them in the vault.
    **Tools**:
    - `list_notes`: List existing notes to avoid duplicates.
    - `read_note`: Read existing notes for context if needed.
    - `create_note`: Create a new plan note in the vault.
    - `update_note`: Update an existing plan if requested.
    - `OutlineAgent`: Generate structured outlines from detailed plans.
    - `SummarizeAgent`: Summarize large tasks or goals.
    - `SearchAgent`: Return relevant information from the web if needed.
    - `DocumentationAgent`: Consult Obsidian documentation if the task is related to Obsidian Vaults.
    - `TemplateAgent`: Format the plan according to a specific template if requested.
    - `LintAgent`: Ensure proper markdown formatting.
    **Guidelines**:
    - Use clear, action-oriented language.
    - Structure plans hierarchically with headings and lists.
    - Use Markdown syntax and use `LintAgent` to check for formatting issues.
    - Estimate timeframes *only* if requested.
    - Avoid making assumptions about user intent.
    - Update an existing plan *only* if requested.
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    - If details are missing (e.g., deadlines, specific tools), 
      *infer* the most logical default based on the project type.
    - If an inference cannot be made, use a placeholder like `[Undetermined Tool]`.
    - Do not halt the process to ask the user for clarification.
    **Workflow**
    1. **Analyze**: Check existing notes (`list_notes`, `read_note`) to understand context if needed.
    2. **Research**: Collect any information required using `SearchAgent` and `DocumentationAgent`.
    3. **Structure**:
       - Use `OutlineAgent` to create the skeleton.
       - Use `SummarizeAgent` to condense main points and its contents into brief overviews for each section.
    4. **Draft & Refine**:
       - Assemble the plan in Markdown.
       - Use `TemplateAgent` if a specific format is requested.
       - Use `LintAgent` to ensure clean Markdown syntax.
    5. **Save**: Create the plan note using `create_note` or update an existing one using `update_note`.
    **Output**:
    - A clear, hierarchical Markdown note.
    - Action-oriented language.
    - Clear deliverables.
    """,
    tools=[list_notes, read_note, create_note, update_note, 
           template_tool, lint_tool, outline_tool, summarize_tool, search_tool, docs_tool],
    output_key="task_plan_note",
)

# Context File Agent

context_file_agent = LlmAgent(
    name="ContextFileAgent",
    model=model_pro,
    description="Create and maintain high-quality context files (e.g., AGENTS.md, GEMINI.md) for LLMs.",
    instruction="""You are a Context File Specialist. 
    **Role**:
    - Your goal is to create or update high-quality documentation and context files (e.g., AGENTS.md, GEMINI.md) for LLMs, 
    following the best practices at https://agents.md/.
    **Tools**:
    - `list_notes`: List existing context files to avoid duplicates.
    - `create_note`: Create new context files in the vault.
    - `update_note`: Update existing context files in the vault.
    - `read_note`: Read existing context files as references or instruction files.
    - `SearchAgent`: Research best practices for context files if needed.
    **Guidelines**:
    - Use `SearchAgent` to research or find examples if needed.
    - Follow best practices from https://agents.md/.
    - Use clear Markdown formatting.
    - Create documentation that helps LLMs understand a project's structure, architecture, and conventions.
    - Keep context concise but comprehensive enough for an LLM to be useful.
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    - If you are missing specific details (like a full directory tree or specific file contents), 
    *infer* what you can from the user's prompt or existing notes.
    - If you cannot infer it, use a clear placeholder (e.g., `` or `[TODO: Add tech stack details]`) 
    and continue generating the file.
    **Workflow**:
    1.  **Assess Information**: 
        - Determine what information is needed to create the context file (e.g., project tree, 
        tech stack, agent definitions).
        - Use `SearchAgent` to research best practices, examples if necessary.
    2.  **Gather Data**:
        - Use `list_notes` to see if relevant documentation already exists in the vault.
        - Use `read_note` if you need to extract information from a README or plan created in a previous steps.
    3.  **Draft Content**: Create the context file using standard sections and best practices (from https://agents.md/):
        - **Project Overview**: High-level summary.
        - **Architecture**: Diagrams or descriptions of how components interact.
        - **Conventions**: Coding style, naming patterns.
        - **Roadmap/Status**: Current state of the project.
    4.  **Save**: Use `create_note` or `update_note` to save the file in the vault.
    **Output**:
    A well-structured context file in markdown format (e.g., AGENTS.md, GEMINI.md).
    """,
    tools=[search_tool, list_notes, read_note, create_note, update_note],
    output_key="context_file"
)

# VS Code Specialist Agent

vscode_specialist_agent = LlmAgent(
    name="VSCodeSpecialist",
    model=model_pro,
    description="Help users create custom agent definitions for VS Code Copilot Custom Agents.",
    instruction="""You are a VS Code Specialist.
    **Role**: Your goal is to help users create custom agent definitions for VS Code Copilot Custom Agents.
    **Tools**:
    - Use `SearchAgent` to verify the latest syntax for VS Code custom agents.
    - Use `list_notes` to check for existing agent definitions.
    - Use `create_note` to save the agent definition.
    - Use `update_note` to modify existing agent definitions.
    - Use `read_note` to read existing agent definitions for reference or other information
    **Guidelines**:
    - Follow the official documentation (https://code.visualstudio.com/docs/copilot/customization/custom-agents).
    - Ensure the agent definitions are clear, concise, and well-structured.
    **CRITICAL RULE: DO NOT STOP TO ASK QUESTIONS.**
    - If you do not know the exact file names in the user's project, use standard wildcards or 
      common names (e.g., `*.py`, `src/*`, `README.md`).
    - Infer the system prompt based on the agent's name (e.g., if the agent is "Python Expert", 
      infer that it should prioritize PEP8 and performance).
    - Do not wait for the user to paste file contents.
    **Workflow**:
    1.  **Analyze Goal**: Understand the user's goal for the new agent (e.g., "Code Reviewer", "Python Expert").
    2.  **Assess Information**:
        - Use `SearchAgent` to verify the latest syntax, examples, and best practices for VS Code custom agents.
        - Use `list_notes` to check for existing agent definitions in the vault.
    3.  **Infer Context**: 
        - If the user references a plan or architecture, use `list_notes` and `read_note` to get details.
        - Otherwise, assume standard industry defaults for that domain.
    4.  **Draft System Prompt**: Write a comprehensive instruction for that agent.
    5.  **Refine**: Ensure clarity, conciseness, and proper structure following official guidelines.
    6.  **Save**: Create a markdown note in the vault containing this definition (e.g., `copilot-agent-python-expert.md`).
    **Output**:
    A structured markdown file that follows the official VS Code Copilot Custom Agents documentation guidelines.
    """,
    tools=[search_tool, list_notes, read_note, create_note, update_note],
    output_key="vscode_agent_definition"
)

print("✅ Specialist Agents built!")

✅ Specialist Agents built!


## 🧠 Section 4: Root Agent

### 4.1: The Orchestrator

The Root Agent is the main entry point that interacts with users. It analyzes user intent, can use tools, delegates to appropriate sub-agents or uses tool agents, and assembles final responses. 

It needs to have a comprehensive understanding of all available tools and agents to effectively manage user requests.

In [15]:
# Define the root agent

root_agent = LlmAgent(
    name="ObsidianAgent",
    model=model_pro,
    instruction="""You are the Obsidian Agent, the root orchestrator for a markdown knowledge base.

    **Role & Responsibilities**:
    - Your main purpose is to analyze the user intent and route requests to the appropriate tools or agents.

    **Tools**:
    A. File Manipulation:
    - `list_notes`: list notes in the vault.
    - `read_note`: read existing notes.
    - `delete_note`: remove notes.
    - `merge_notes`: combine notes.
    B. Tool Agents (agents that understand user intent):
    - `ReviewAgent`: to review existing notes.
    - `DocumentationAgent`: for Obsidian documentation questions.
    - `SearchAgent`: for web research.
    - `QuickLogger`: to create, append or update notes.

    **Sub-Agents**:
    - `ContextFileAgent`: to create project context files.
    - `VSCodeSpecialist`: to create VS Code Copilot agent definitions.
    - `ExistingContentEditor`: to edit existing notes.
    - `NewContentCrafter`: to create new notes.
    - `TaskPlanner`: to create structured plans.

    **Guidelines and Workflow**:
    - Carefully analyze user intent to determine if they want to create new content, edit existing notes, or simply ask a question.
    - Use the *Routing Table* to determine the appropriate agent/tool for each task (consider *a* and *b* IF *create* or *edit*):
        a. To CREATE new documents, decide whether to use `QuickLogger` (simple tasks, you can provide the full content)
        or `NewContentCrafter` for complex tasks (slow iterative write).
        b. To EDIT documents decide, based on the complexity of the request, whether to append to document or replace content
        using `QuickLogger` (for small documents, you can provide the updated content) or, call `ExistingContentEditor` (slow iterative rewrite).
    - To MERGE/COMBINE documents (not just appending one note to another), use `merge_notes` and provide the merged document to `ExistingContentEditor`.
    - For simple questions or research, use `SearchAgent` and `DocumentationAgent` (call `QuickLogger` to create a file if needed).
    - Always verify file targets with `list_notes` and ask user for confirmation if files exists with similar names.
    - Ensure user confirmation before using destructive actions (e.g. delete, merge).

    **Routing Table**:

    | User Intent                                                | Agent/Tool                     |
    |------------------------------------------------------------|--------------------------------|
    | *Quick* logs, journals, lists (create, update or append)   | `QuickLogger`                  |
    | *Create* refined new documents, blogs, research            | `NewContentCrafter`            |
    | *Edit*, refine, or rewrite existing documents              | `ExistingContentEditor`        |
    | Create or update *Developer* Context (AGENTS.md, README)   | `ContextFileAgent`             |
    | Create *VS Code Copilot* Custom Agent definitions          | `VSCodeSpecialist`             |
    | *Plan* structured goals or project roadmaps                | `TaskPlanner`                  |
    | *Obsidian documentation* questions                         | `DocumentationAgent`           |
    | *Search* information, facts or examples and templates      | `SearchAgent`                  |
    """,
    
    tools=[
        list_notes, read_note, delete_note, merge_notes, # Function Tools
        review_tool, log_tool, docs_tool, search_tool, # Tool Agents
    ],
    sub_agents=[task_plan_agent, context_file_agent, vscode_specialist_agent, # Specialized Agents
                existing_content_editor, new_content_crafter], # Workflow Agents
    output_key="root_response",
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


### 4.2: Configure Apps

In ADK, the App class is designed to manage a collection of agents grouped by a root agent. It separates the concerns of an agent workflow's overall operational infrastructure from individual agents task-oriented logic, and provides a higher-level abstraction for managing agents, centralizing configuration, manage lifecycle, defining an explicit state scope and establishing a deployable unit.

We'll define the default app as follows, with a previously defined list of plugins that could be used later.

In [16]:
# Define a default app that uses the root agent

obsidian_agent_app = App(
    name='obsidian_assistant',
    root_agent=root_agent,
)

print("✅ Obsidian Agent App created!")

✅ Obsidian Agent App created!


### 4.3 Plugins and Enhanced Apps

From a practical perspective, the App class is used to configure context caching and compression, as well as other features to enhance and control the agent workflow like the plugins.

**Plugins** functionality builds on **Callbacks**, which is a key design element of the ADK's extensible architecture, and it is possible to create custom plugins or add the prebuilt ones to the apps.

In [17]:
# Initialize plugins variable
app_plugins = []

print("✅ Plugins initialized.")

✅ Plugins initialized.


## 📚 Section 5: Runtime, Sessions and Memory

There are three concepts to understand within agents conversations: **Session**, the current conversation thread, **State**, the data within the conversation, and **Memory**, searchable, cross-session information. 

Agents previously defined are executed in the **Runtime** using the **Runner** class, that operates on an **Event** Loop, managing the lifecycle of the agent, handling incoming requests, and coordinating interactions between the agent, its tools, and services like session and memory.

The **Session Service** stores the conversation history for the current interaction. It allows the agent to remember what was just said. The **Memory Service** allows the agent to recall facts or preferences from past interactions, storing knowledge across different sessions, and it can be used to build a more personalized experience for the user, even including external data sources.

In this section, we will set up two **Runners** with different configurations for our agent:

- **Runner with Session Service:** This runner will utilize the session service for managing short-term conversation context. It will use `InMemorySessionService` to handle conversation history within a session and can handle multiple concurrent sessions per user (different session names).
- **Runner with Memory Service:** This runner will utilize both the session service and the memory service for enhanced memory capabilities. It will use `InMemoryMemoryService` to handle persistent facts across sessions, user preferences, and knowledge retention.

The `InMemoryMemoryService` is suitable for prototyping and testing purposes. For production use cases, we should consider the builtin Vertex AI Memory Service or other scalable memory solutions.

### 5.1: Implementing Statefulness

For stateless interactions, we can use the session service to maintain context within a single session. For stateful interactions, we should leverage the memory service to retain information across multiple sessions and enable enhanced capabilities of our apps related to plugin usage that require `state` management.

In [18]:
# Initialize the global session and memory services

print("🔄 Starting session services...")
session_service = InMemorySessionService()
memory_service = InMemoryMemoryService()
print("✅ In-memory session and memory services have been reset.")

🔄 Starting session services...
✅ In-memory session and memory services have been reset.


The next step is to implement the **Runtime** using the defined **Runners** (we can define runners for different purposes) to execute the Obsidian Agent **Apps** (we could use any `app` for each runner) within a **Session** service context (`session_service` is another argument for the `Runner` class).

In [19]:
# Create a Runner with session service with default app configuration
runner = Runner(
    app=obsidian_agent_app,
    session_service=session_service,
)

print("✅ Runner initialized and ready to execute!")

✅ Runner initialized and ready to execute!


In [20]:
# Create a Runner with session service and memory service with enhanced app configuration
runner_memory = Runner(
    app=obsidian_agent_app,
    session_service=session_service,
    memory_service=memory_service # Enable memory service
)

print("✅ Runner with memory service initialized and ready to execute!")

✅ Runner with memory service initialized and ready to execute!


### 5.2: Memory retrieval (load and preload)

ADK provides two built-in tools for memory retrieval:

- **`load_memory` (Reactive)**: Agent decides when to search memory

- **`preload_memory` (Proactive)**: Automatically searches before every turn

We will integrate by default the more efficient `load_memory` tool into the root agent tools, and we could swap it for `preload_memory` if we want to test proactive memory retrieval.

In [21]:
# Add load memory tool into root agent's toolset
if load_memory not in root_agent.tools:
    root_agent.tools.insert(0, load_memory)

print("✅ Load Memory tool added to Root Agent's toolset.")

✅ Load Memory tool added to Root Agent's toolset.


### 5.3: Enhanced Apps with Event Compacting and Context Caching

Event compacting is a technique used to optimize memory usage by reducing the amount of historical data retained in memory. In the context of our agent, this means we can keep the most relevant parts of the conversation while discarding less important details.

Context caching is another powerful feature that allows the agent to store and reuse relevant information from previous interactions. By enabling context caching in our app configuration, we can significantly improve the agent's performance and responsiveness.

By leveraging event compacting and context caching, we can enhance the memory capabilities of our agent while keeping resource usage in check. This is particularly important for long-running sessions or when dealing with large volumes of data.

In [22]:
# Import necessary library and hide experimental warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*EXPERIMENTAL.*')

# Configure Events Compaction
events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,  # Trigger compaction every 3 invocations
        overlap_size=2,  # Keep 2 previous turns for context
)

# Configure Context Caching
context_cache_config=ContextCacheConfig(
        min_tokens=10000,    # Minimum tokens to trigger caching
        ttl_seconds=1800,    # Store for up to 30 minutes
        cache_intervals=1,  # Refresh cache after each use
)

print("✅Events Compaction and Context Cache configured.")

✅Events Compaction and Context Cache configured.


### 5.4: Modularity

By breaking down an agent into smaller, reusable components, we can improve maintainability and scalability. We can define the components of an App previously or use components defined within other Apps defined in the system.

In [23]:
# Define an app with the new configurations
obsidian_enhanced_app = App(
    name='obsidian_enhanced_assistant',
    root_agent=root_agent,
    events_compaction_config=events_compaction_config,  
    context_cache_config=context_cache_config,
    plugins=app_plugins
)

# Create a new runner for our complex app
runner_enhanced = Runner(
    app=obsidian_enhanced_app, session_service=session_service, memory_service=memory_service
)

## 👁️ Section 6: Observability

As learned in the 5-day AI Agents Intensive Course, we should keep in mind the three pillars of observability: **logging**, **tracing**, and **monitoring**.

### 6.1: Logging and Tracing with built-in Plugin

In this section, we'll integrate an ADK's built-in **logging plugin** to capture detailed text logs of agents interactions, including intermediate requests, model responses, outputs, and tool usage; allowing us to trace the agent's behavior.

The `LoggingPlugin` handles the heavy lifting of tracing agent execution. This plugin automatically hooks into the runner's event loop to capture and display:

LLM Requests & Responses: See exactly what the model is thinking.

Tool Calls: Observe inputs and outputs for tools like list_notes or read_note.

Agent Handoffs: Track the flow between the Root Agent and Sub-Agents.

For testing and prototyping, exists a subclass of Runner, `InMemoryRunner`, designed to get started quickly with minimal setup. This works the same way as an app which takes the plugin parameter and handle directly an agent, instead of `app`, the parameter required by the common `Runner`.

In [24]:
# Create a single instance of LoggingPlugin
logging_plugin = LoggingPlugin()

# Create the InMemoryRunner with the logging plugin
runner_logging = InMemoryRunner(
    root_agent,
    plugins=[logging_plugin],  # Add the plugin
)

By using the `LoggingPlugin`, it becomes unnecessary to manually construct logs using standard logging library, however, the ADK's underlying GenAI library typically warns when a model response contains "non-text parts" (i.e., Function Calls or structured tool outputs). Since our architecture relies on agents using tools, these warnings are expected behavior and actually indicate the system is working correctly. We could suppress them here to keep the notebook output clean and focused on the agent's logic, and the `logging` module is necessary to access the warnings logging level to silence them.

In [25]:
# Import logging module
import logging

# Silence the specific warning from the Google GenAI library
logging.getLogger("google_genai.types").setLevel(logging.ERROR)

print("✅ Non-text warnings suppressed.")

✅ Non-text warnings suppressed.


### 6.2: Monitoring & Metrics

While Logging gives us the story of what happened, monitoring focuses on how well it happened. For this Obsidian Agent, we will implement a local monitoring check. We will extract metrics to measure latency (session duration) and cost (token usage) from our agent sessions.

We would also filter final responses based on a whitelist of visible agents, to avoid extending the output with intermediate steps from sub-agents, that could be supervised in the logging trace.

#### The whitelist

In [26]:
# Define the whitelist of visible agents for the output filter including root agent
white_list = [root_agent.name]

# Add the specialized Sub-Agents
white_list.extend([
    task_plan_agent.name, 
    context_file_agent.name, 
    vscode_specialist_agent.name
])

# Add the "final step" agents of sequential workflows
white_list.extend([
    publish_agent.name, 
    updater_agent.name
])

print(f"👀 Showing outputs for: {white_list}")

👀 Showing outputs for: ['ObsidianAgent', 'TaskPlanner', 'ContextFileAgent', 'VSCodeSpecialist', 'PublishAgent', 'UpdaterAgent']


#### Helper function to run sessions
We'll define a helper function that manages a complete conversation session, handling session creation/retrieval, query processing, and response streaming. The helper function will support both single queries and multiple queries in sequence and includes memory handling based on the runner configuration.

In [27]:
# Import time module for monitoring session duration
import time

# Define helper function to run a session with the given runner, user queries and session name
async def run_session(
    runner: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
    visible_agents: list[str] = None, # Final responses whitelist parameter
) -> None: # This is important to include the type hint for None return to avoid duplication in output cells
    
    # Set visible agents to white_list as default argument
    try:
        if visible_agents is None:
            visible_agents = white_list
    except NameError:
        print("ℹ️ Info: white_list not defined, using all agents for output visibility.")
        visible_agents = None # Fallback in case white_list is not defined

    print("--- Running session ---\n")

    # Use the services attached to the runner instance
    session_service = runner.session_service
    memory_service = runner.memory_service

    # Define app/user names
    app_name = runner.app_name
    session_id = session_name
    user_id = USER_NAME

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=user_id, session_id=session_id
        )
        print(f"✅ Session {session_name} created for: {app_name}\n")
    except Exception:
        # If creation fails (e.g., session already exists), try to retrieve it.
        # Log the exception for debugging.
        print("ℹ️ Session exists. Attempting to retrieve existing session...\n")
        session = await session_service.get_session(
            app_name=app_name, user_id=user_id, session_id=session_id
        )
        print(f"✅ Session {session_name} retrieved for: {app_name}\n")

    # Start Timer
    start_time = time.time()
    print("--- Monitoring session ---\n")
       
    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if isinstance(user_queries, str):
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query_text in user_queries:
            print(f"User >>> {query_text}\n")

            # Convert the query string to the ADK Content format
            query = types.Content(role="user", parts=[types.Part(text=query_text)])

            # Initialize full response and token usage accumulators
            full_response_text = ""
            token_count = 0

            # Stream and process events from the agent event generator
            async for event in runner.run_async(user_id=session.user_id, session_id=session.id, new_message=query):
                agent_name = getattr(event, "author", "System") # Extract agent name

                # Content handling
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        text_part = getattr(part, 'text', None)
                        if text_part:
                            incoming_text = text_part
                            if event.partial:
                            # If partial is True, we append the text to the existing response
                                full_response_text += incoming_text
                            else:
                            # If partial is False, the text is the *entire* response for that turn
                                full_response_text = incoming_text
                        else:
                            incoming_text = "" # Prevent NoneType errors

                # Metrics handling
                if event.usage_metadata:
                    # We grab the total from the specific response event
                    token_count += event.usage_metadata.total_token_count

                # Check if specific visibility rules apply
                if visible_agents is None or agent_name in visible_agents:
                    # Print final responses detected
                    if event.is_final_response():
                        response = full_response_text.strip()
                        print(f"\n--- Final output detected from [{agent_name}] ---\n")
                        print(f"Agent >>> {response}\n")
                        print("--- End of response ---\n")
                
            # Memory Handling
            if runner.memory_service is not None:
                # Retrieve the completed session from session service
                completed_session = await session_service.get_session(app_name=app_name, user_id=user_id, session_id=session.id)
                # Add this session's content to the memory service
                print("ℹ️ Adding session to memory...\n")
                await memory_service.add_session_to_memory(completed_session)
                print(f"✅ Session {completed_session.id} added to memory\n")
            
            # Stop Timer
            end_time = time.time()
            duration = round(end_time - start_time, 2)

            # Print results
            print(f"--- Session Metrics ---\n\nDuration: {duration}\nToken Usage: {token_count}\n")
            print("--- Session End ---\n")        

print("✅ Run session function defined.")

✅ Run session function defined.


### 6.3 Testing the Observability Features

We'll start with a simple tests for different functionalities with the following code:

 ```python
 await run_session(
     runner_name,
     "Prompt to agent",
     session_name="session_name"
 )
 ```

 If any test is rerun, the session_service will retrieve the existing session, preserving the conversation history; however, the memory_service will not be used in this runner configuration, so no memory will be retained across sessions.

In [28]:
# Observability Test: Simple vault query
await run_session(
    runner_logging,
    "There are notes in my vault?",
    session_name="test",
)

--- Running session ---

✅ Session test created for: InMemoryRunner

--- Monitoring session ---

User >>> There are notes in my vault?

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-7e7bc72f-3838-4426-a61c-1cbd1f1936ae
[logging_plugin]    Session ID: test
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'There are notes in my vault?'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-7e7bc72f-3838-4426-a61c-1cbd1f1936ae
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-7e7bc72f-3838-4426-a61c-1cbd1f1936ae
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model: gemini-2.5-pro
[logging_plugin]    Agent: ObsidianAgent
[logging_plugin]    System Instruction: 'You are the Obsidian Agent, the root orchestr

To include the logging functionality in production, as the InMemoryRunner is intended for testing and prototyping, we would need to modify the app to include the LoggingPlugin and re-initialize the runners.

In [29]:
# Add the logging plugin to the Obsidian Agent App
if logging_plugin not in app_plugins:
    app_plugins.append(logging_plugin)

print("✅ Logging Plugin added to Obsidian Agent App.")

# Ensure the app plugins are updated
obsidian_agent_app.plugins = app_plugins
obsidian_enhanced_app.plugins = app_plugins

# Re-initialize the runners so they pick up the new plugin configuration
runner = Runner(
    app=obsidian_agent_app,
    session_service=session_service,
)

runner_memory = Runner(
    app=obsidian_agent_app,
    session_service=session_service,
    memory_service=memory_service
)

runner_enhanced = Runner(
    app=obsidian_enhanced_app, 
    session_service=session_service, 
    memory_service=memory_service
)

print("✅ Runners re-initialized with LoggingPlugin enabled.")

✅ Logging Plugin added to Obsidian Agent App.
✅ Runners re-initialized with LoggingPlugin enabled.


## 🚀 Section 7: Run Agent Sessions

Now we'll combine everything we've prepared (tools, agents, app, runner, configurations) to execute the agent. Each run will focus on a specific capability of the Obsidian Agent, such as creating notes, reading content, or using tools.

### 7.1 Recap and Execution Plan

We'll first recap on available tools and agents and then will prepare a list of run cases and observe the agent's behavior through the logging outputs, and performance through the monitoring metrics duration and token usage.

#### Tools and Agents

A recap of the Tools and Agents we have defined for the Obsidian Agent:

**Function Tools:**

- **File Manipulation Tools**: `create_note`, `read_note`, `update_note`, `delete_note`, `list_notes`, `append_to_note`, `merge_notes`.

**Tool Agents (AgentTool):**

- **Content**: `summarize_tool`, `key_points_tool`, `outline_tool`, `breakdown_tool`, `review_tool`. 

- **Formatting**: `lint_tool`, `template_tool`, `style_tool`.

- **Web Tools**: `docs_tool`, `search_tool`.

- **Utility**: `quick_log_tool` (The Fast-Track Logger)

**Sub-Agents:**

- **Workflow Agents**: 
    - `NewContentCrafter` (The Smart Writer)
    - `ExistingContentEditor` (The Wise Editor)

- **Specialized Agents**: 
    - `TaskPlanner`
    - `ContextFileAgent`
    - `VSCodeSpecialist`

#### Run Plan

It is interesting to test the performance of each runner by executing a simple query. Before this performance test, we'll execute the following query sessions with different configurations to try our Obsidian Agent.

##### Single Queries:

| Session Name       | Description                              | 
|--------------------|------------------------------------------|
| web_query          | Create a simple note from web content.   |
| quick_log          | Append content to an existing note.      |
| note_craft         | Craft a new note with specific content.  |
| edit_note          | Update an existing note in the vault.    |
| plan_task          | Plan a multi-step complex objective.     |

##### Multi-Step and Multi-Threaded Memory Session:

| Session Name            | Description                                          |
|-------------------------|------------------------------------------------------|
| research_step_(1-3)     | An investigation that involves multiple web queries. |

##### The Proof of Concept:

A complex session that involves multiple steps and the collaboration of various specialized agents. The perfect scenario could be a software development project where the plan task agent will break down the objective into manageable steps, the context file agent will create necessary context files and the README, and the VSCode specialist will assist in creating a custom agent (different from built-in Ask, Plan, Edit and Agent modes).

| Session Name                | Description                                                 |
|-----------------------------|-------------------------------------------------------------|
| test_project_setup          | Set up a new project with multiple notes and complex tasks. |

**Note:** the API key limits (specially free accounts) may affect the execution of multiple sessions in a short period. If you encounter rate limit errors, consider adding delays between session executions or reducing the number of sessions run consecutively.

### 7.2: Single Queries

Now we'll execute the single query sessions defined in our test plan. Each session will be run with different configurations to evaluate the performance and capabilities of the Obsidian Agent.

#### Web Query

In [30]:
await run_session(
    runner,
    "Create a note with the most important news headlines of the week in tech related to Google",
    session_name="test_query"
)

--- Running session ---

✅ Session test_query created for: obsidian_assistant

--- Monitoring session ---

User >>> Create a note with the most important news headlines of the week in tech related to Google

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-8810e8bb-35da-47e4-9d01-fd18ad9b66fc
[logging_plugin]    Session ID: test_query
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Create a note with the most important news headlines of the week in tech related to Google'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-8810e8bb-35da-47e4-9d01-fd18ad9b66fc
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-8810e8bb-35da-47e4-9d01-fd18ad9b66fc
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model

#### Quick Log

In [31]:
await run_session(
    runner,
    "Add guanciale and pecorino cheese to my shopping list (create the note if it does not exist).",
    session_name="test_quick_log_append"
)

--- Running session ---

✅ Session test_quick_log_append created for: obsidian_assistant

--- Monitoring session ---

User >>> Add guanciale and pecorino cheese to my shopping list (create the note if it does not exist).

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-2302d62f-f098-4946-b86f-19fab729ec87
[logging_plugin]    Session ID: test_quick_log_append
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Add guanciale and pecorino cheese to my shopping list (create the note if it does not exist).'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-2302d62f-f098-4946-b86f-19fab729ec87
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-2302d62f-f098-4946-b86f-19fab729ec87
[logging_plugin] 🧠 LLM REQUE

#### Note Creation

In [32]:
await run_session(
    runner,
    "Create a new note with exploring the benefits and possibilities of using Obsidian for knowledge management.",
    session_name="test_note_crafting"
)

--- Running session ---

✅ Session test_note_crafting created for: obsidian_assistant

--- Monitoring session ---

User >>> Create a new note with exploring the benefits and possibilities of using Obsidian for knowledge management.

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-bf87eada-7d49-4ee2-bf0c-a5f6ecb07e36
[logging_plugin]    Session ID: test_note_crafting
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Create a new note with exploring the benefits and possibilities of using Obsidian for knowledge management.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-bf87eada-7d49-4ee2-bf0c-a5f6ecb07e36
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-bf87eada-7d49-4ee2-bf0c-a5f6ecb07e36
[loggi

#### Note Editing

In [33]:
await run_session(
    runner,
    "Update my note about Google tech news and add a detailed section about Antigravity IDE.",
    session_name="test_note_editing"
)

--- Running session ---

✅ Session test_note_editing created for: obsidian_assistant

--- Monitoring session ---

User >>> Update my note about Google tech news and add a detailed section about Antigravity IDE.

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-ef9371c6-692d-4263-8465-44d5bd35383d
[logging_plugin]    Session ID: test_note_editing
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Update my note about Google tech news and add a detailed section about Antigravity IDE.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-ef9371c6-692d-4263-8465-44d5bd35383d
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-ef9371c6-692d-4263-8465-44d5bd35383d
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin] 

#### Plan Task

In [34]:
await run_session(
    runner,
    "Help me create a plan to learn the basics of Python for ML and AI. Break it into weekly goals.",
    session_name="test_task_planning"
)

--- Running session ---

✅ Session test_task_planning created for: obsidian_assistant

--- Monitoring session ---

User >>> Help me create a plan to learn the basics of Python for ML and AI. Break it into weekly goals.

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-860b0e0e-95d7-45f7-8423-0e4410fc08d9
[logging_plugin]    Session ID: test_task_planning
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Help me create a plan to learn the basics of Python for ML and AI. Break it into weekly goals.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-860b0e0e-95d7-45f7-8423-0e4410fc08d9
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-860b0e0e-95d7-45f7-8423-0e4410fc08d9
[logging_plugin] 🧠 LLM REQUEST
[

### 7.3: Multi-Step Session

We'll execute a multi-step session that involves memory usage. This session will test the agent's ability to handle complex workflows and retain information across multiple interactions.

#### Research Workflow

##### Step 1: Initial topic exploration

In [35]:
print("--- Step 1: Initial Research ---")
await run_session(
    runner_memory,
    "Find information about 'Agentic Design Patterns', specifically focusing on 'Reflection' and 'Tool Use'.",
    session_name="research_step_1"
)

--- Step 1: Initial Research ---
--- Running session ---

✅ Session research_step_1 created for: obsidian_assistant

--- Monitoring session ---

User >>> Find information about 'Agentic Design Patterns', specifically focusing on 'Reflection' and 'Tool Use'.

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-b9b1f687-56e3-4796-90b7-b8e8f67ba54e
[logging_plugin]    Session ID: research_step_1
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Find information about 'Agentic Design Patterns', specifically focusing on 'Reflection' and 'Tool Use'.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-b9b1f687-56e3-4796-90b7-b8e8f67ba54e
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-b9b1f687-56e3-4796-90b7-

##### Step 2: Follow-up (New Session)

In [36]:
print("\n--- Step 2: Follow-up ---")
await run_session(
    runner_memory,
    "Compare those two patterns we discussed earlier. Which one is generally considered more complex to implement?",
    session_name="research_step_2"
)


--- Step 2: Follow-up ---
--- Running session ---

✅ Session research_step_2 created for: obsidian_assistant

--- Monitoring session ---

User >>> Compare those two patterns we discussed earlier. Which one is generally considered more complex to implement?

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-01498948-1033-4e98-bdfe-ef7ea499c0c6
[logging_plugin]    Session ID: research_step_2
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Compare those two patterns we discussed earlier. Which one is generally considered more complex to implement?'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-01498948-1033-4e98-bdfe-ef7ea499c0c6
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-01498948-1033-4e98

##### Step 3: Synthesis

In [37]:
print("\n--- Step 3: Note Creation (New Session) ---")
await run_session(
    runner_memory,
    "Create a note titled 'Agentic Patterns Summary.md' summarizing our discussion.",
    session_name="research_step_3",
)


--- Step 3: Note Creation (New Session) ---
--- Running session ---

✅ Session research_step_3 created for: obsidian_assistant

--- Monitoring session ---

User >>> Create a note titled 'Agentic Patterns Summary.md' summarizing our discussion.

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-ed8f7498-a41d-4e66-847b-75d6e169e62a
[logging_plugin]    Session ID: research_step_3
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'Create a note titled 'Agentic Patterns Summary.md' summarizing our discussion.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-ed8f7498-a41d-4e66-847b-75d6e169e62a
[logging_plugin]    Starting Agent: ObsidianAgent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ObsidianAgent
[logging_plugin]    Invocation ID: e-ed8f7498-a41d-4e66-847b-75d6e169e62a
[logging_plugin] 🧠 LLM RE

### 7.4: Project setup scenario

For this complex scenario, we will simulate the setup of a new project using the Obsidian Agent. The agent will need to create multiple content pieces, organize them, plan a project, write the documentation, including the context file for agents that work within the project, and a custom VSCode agent mode file. This will demonstrate the agent's ability to manage a comprehensive task involving various components of the Obsidian vault.

##### Swap the memory retrieval strategy

In [38]:
# Switch from load memory (reactive) to preload memory (proactive)
if load_memory in root_agent.tools:
    root_agent.tools.remove(load_memory) # Remove load memory from root agent's toolset
    print("✅ Load Memory tool removed from Root Agent's toolset.")
if preload_memory not in root_agent.tools:
    root_agent.tools.insert(0, preload_memory) # Add preload memory to root agent's toolset
    print("✅ Preload Memory tool added to Root Agent's toolset.")

✅ Load Memory tool removed from Root Agent's toolset.
✅ Preload Memory tool added to Root Agent's toolset.


##### Project definition steps

1. Brainstorm the project idea.
2. Create a project plan breaking down the tasks.
3. Create an initial README.md file documenting the project.
4. Create a context file AGENTS.md for the agents working on this project.
5. Create a VS Code Copilot custom agent definition for a "Python Memory Expert" agent.
    - The agent should be instructed to prioritize memory efficiency in Python applications and notebooks.
    - This mode should be strictly optimized for refactoring and modularization.
6. Write an Executive Summary of the project plan and documentation created.

To include the logging functionality in production, as the InMemoryRunner is intended for testing and prototyping, we would need to modify the app to include the LoggingPlugin and re-initialize the runners.

In [39]:
# Define prompts for project setup tasks
prompt1 = """I have an idea for a 'Crypto Sentiment Analysis' tool. 
        I want to ingest data from Reddit, use a simple NLP model to analyze sentiment, 
        and correlate it with price charts. Help me brainstorm the core features and a 
        modern Python tech stack for this."""

prompt2 = """Based on the brainstorming, create a detailed Project Plan with clear implementation steps.
         Focus on a modular architecture."""

prompt3 = """Create an initial README.md file documenting the project structure and setup."""

prompt4 = """Create a context file AGENTS.md that explains the project to other AI agents who might work 
        on this codebase."""

prompt5 = """Create a VS Code Copilot custom agent definition for a "NLP Expert" agent.
        The agent should be instructed to prioritize memory efficiency in Python."""

prompt6 = """Generate a concise Executive Summary of everything we accomplished in this session. 
        List the files created and the next immediate steps for the developer."""

##### Project Workflow

In [40]:
# Run the sessions sequentially to simulate a project setup workflow
session = "project_setup_test"
for prompt in [prompt1,prompt2,prompt3,prompt4,prompt5,prompt6]:
    print(f"\n--- Setup a project test - Session: {session} ---")
    await run_session(
        runner_enhanced,
        prompt,
        session_name=session
    )


--- Setup a project test - Session: project_setup_test ---
--- Running session ---

✅ Session project_setup_test created for: obsidian_enhanced_assistant

--- Monitoring session ---

User >>> I have an idea for a 'Crypto Sentiment Analysis' tool. 
        I want to ingest data from Reddit, use a simple NLP model to analyze sentiment, 
        and correlate it with price charts. Help me brainstorm the core features and a 
        modern Python tech stack for this.

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-b3472bdc-b50e-4590-a51e-e0d628f742f7
[logging_plugin]    Session ID: project_setup_test
[logging_plugin]    User ID: Pau
[logging_plugin]    App Name: obsidian_enhanced_assistant
[logging_plugin]    Root Agent: ObsidianAgent
[logging_plugin]    User Content: text: 'I have an idea for a 'Crypto Sentiment Analysis' tool. 
        I want to ingest data from Reddit, use a simple NLP model to analyze sentiment, 
        and correlate it with price chart

## 🧪 Section 8: Evaluate Agents

The ADK offers different ways for evaluating agents, with the Web-based UI (adk web), using test files (pytest) or Evalset schema file (adk eval CLI command). Agent evaluation can be broken down into **Evaluating Trajectory and Tool Use** (analyzing the steps an agent takes to reach a solution, including tool selection, strategies, and the efficiency of its approach) and **Evaluating the Final Response** (assessing the quality, relevance, and correctness of the agent's final output).

We've implemented a **Qualitative and Intrinsic Evaluation** for the Obsidian Agent. Instead of relying on automated metrics, we included self-evaluation tools in the agent runtime architecture, like the `review_tool` to act as an internal judge (without being the same model instance). By adopting a Critic-Refiner pattern, the agent evaluates its own output against the user's requirements before finalizing the response, ensuring high-quality zero-shot performance.

It is prioritized to conduct human evaluation to capture nuanced aspects of the agent's behavior such as the coherence of note content, adherence to known best practices, and overall user satisfaction.

By executing specific test scenarios (Section 7), tracing the agents behavior within the logging output and verifying the correctness of the generated Markdown files, we can ensure a comprehensive qualitative evaluation.

#### What's in the Vault?

Let's inspect the final artifacts generated by the agent in our vault during the test sessions to verify the results of a single notebook run.

In [41]:
# Check what was created in the vault
print("📋 Vault Contents:")
print("=" * 50)

notes = list_notes()
if notes:
    for note in notes:
        print("-" * 50)
        print(f"\n📄 {note}\n")
        content = read_note(note)
        if content.get("status") == "success":
            note_content = content.get("content", "")
            # Show first 200 characters
            preview = note_content[:200] + "..." if len(note_content) > 200 else note_content
            print(f"{'-' * 18} Note preview {'-' * 18}\n\n{preview}\n")
        print("-" * 50)
else:
    print("No notes found in vault.")
print("=" * 50)

📋 Vault Contents:
--------------------------------------------------

📄 Agentic Patterns Summary.md

------------------ Note preview ------------------

# Summary of Agentic Design Patterns

This note summarizes the key differences between the 'Reflection' and 'Tool Use' agentic design patterns.

## Key Takeaways

- **Tool Use**: This pattern is gener...

--------------------------------------------------
--------------------------------------------------

📄 Crypto Sentiment Analysis Tool Brainstorming.md

------------------ Note preview ------------------

This document outlines the core features and a potential Python-based tech stack for a crypto sentiment analysis tool that ingests data from Reddit, performs NLP for sentiment analysis, and correlates...

--------------------------------------------------
--------------------------------------------------

📄 shopping list.md

------------------ Note preview ------------------

- guanciale
- pecorino cheese

-------------------------

**Note:** Make sure to explore the notes in your notebook's output directory to see the results of the agent's work. And the content of the notes created could be interesting to review! Feel free to copy the notebook and build the agent with ADK to use it in your vaults

## 🔮 Next Steps

Here are some potential next steps to further enhance an Obsidian Agent like this:

- Implement LLM-as-Judge evaluation for automated assessment of agent performance.
- Deploy the Obsidian Agent in a real Obsidian vault to test its capabilities with actual user data.
- Explore more advanced Obsidian features like backlinks, graph view, and plugins.
- Integrate MCP servers for enhanced data processing and retrieval, and explore importing/exporting capabilities.
- Implement advanced memory and knowledge management techniques, such as vector databases.
- Integrate additional observability tools for deeper insights into agent performance and behavior.

I will be glad to hear your feedback. If you go further with this project and the proposed enhancements, please share your results!